# Data Versioning & Data Lineage

Let's see how to version data and keep track of the processing that produces them, with [DVC](https://dvc.org/). DVC separates data versioning, git tracking their metadata (`.dvc` files), from data storage.

The steps are those of a real project: version the data, change them, go back to a version, describe their processing in a pipeline, then get everything back on another machine. So that the demo runs without an account, a bare git repository on this machine, `/tmp/remote.git`, stands for a remote repository (GitHub, GitLab, DagsHub…), and a directory, `/tmp/dvc-store`, for remote data storage (S3, Google Cloud Storage…).

## Setup

This cell installs DVC and its S3 extension, then moves into the project's directory, `~/dvc-project`.

In [1]:
!apt install tree
!pip install dvc dvc-s3
!mkdir -p ~/dvc-project
%cd ~/dvc-project




tree is already the newest version (2.1.1-2ubuntu3.24.04.2).
0 upgraded, 0 newly installed, 0 to remove and 126 not upgraded.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 4.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... - \ done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.6/111.6 kB 6.5 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of fsspec[http] to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 470.1/470.1 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.6/102.6 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.3/79.3 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 1. Create the git repository

The project's git repository, linked to the “remote” one.

In [2]:
!git init --bare -b main /tmp/remote.git
!git init -b main
!git config user.name "Jeanne Durant"
!git config user.email "jeanne@durant.fr"
!git remote add origin /tmp/remote.git

Initialized empty Git repository in /tmp/remote.git/
Initialized empty Git repository in /root/dvc-project/.git/


## 2. Set up DVC and its remote storage

`dvc init` sets up the repository, and `dvc remote add -d` declares the default remote storage, the one `dvc push` and `dvc pull` use without `-r`. With DagsHub, `dvc remote modify` would add the server's address and, with `--local`, the credentials, in `.dvc/config.local`, which `.dvc/.gitignore` keeps out of git.

In [3]:
!dvc init
!dvc remote add -d origin /tmp/dvc-store

Initialized DVC repository.

You can now commit the changes to git.

+---------------------------------------------------------------------+
|                                                                     |
|        DVC has enabled anonymous aggregate usage analytics.         |
|     Read the analytics documentation (and how to opt-out) here:     |
|             <https://dvc.org/doc/user-guide/analytics>              |
|                                                                     |
+---------------------------------------------------------------------+

What's next?
------------
- Check out the documentation: <https://dvc.org/doc>
- Get help and share ideas: <https://dvc.org/chat>
- Star us on GitHub: <https://github.com/treeverse/dvc>
Setting 'origin' as a default remote.


In [4]:
!cat .dvc/config .dvc/.gitignore

[core]
    remote = origin
['remote "origin"']
    url = /tmp/dvc-store
/config.local
/tmp
/cache


In [5]:
!git add .dvc
!git commit -m "Initialize DVC"
!git push origin main

[main (root-commit) 753613a] Initialize DVC
 3 files changed, 10 insertions(+)
 create mode 100644 .dvc/.gitignore
 create mode 100644 .dvc/config
 create mode 100644 .dvcignore
Enumerating objects: 6, done.
Counting objects: 100% (6/6), done.
Delta compression using up to 8 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (6/6), 535 bytes | 535.00 KiB/s, done.
Total 6 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
To /tmp/remote.git
 * [new branch]      main -> main


## 3. Retrieve a first version of the data

[`dvc get`](https://dvc.org/doc/command-reference/get) downloads a file from a git or DVC repository, and `dvc add` has DVC track it: it copies it into the cache, writes `data.csv.dvc` and adds `data.csv` to `.gitignore`.

In [6]:
!dvc get https://github.com/shuuchuu/datasets wikipedia-movie-plots/wiki_movie_plots_deduped.csv -o data.csv

In [7]:
!dvc add data.csv

⠋ Checking graph
Adding...                                                                       
!
                                                                                
!
  0% Checking cache in '/root/dvc-project/.dvc/cache/files/md5'| |0/? [00:00<?, 
                                                                                
!
  0%|          |Adding data.csv to cache              0/1 [00:00<?,     ?file/s]
                                                                                
!
  0%|          |Checking out /root/dvc-project/data.cs0/1 [00:00<?,    ?files/s]
100% Adding...|████████████████████████████████████████|1/1 [00:00,  1.81file/s]

To track the changes with git, run:

	git add data.csv.dvc .gitignore

To enable auto staging, run:

	dvc config core.autostage true


In [8]:
!cat data.csv.dvc
!tree .dvc/cache

outs:
- md5: a546111eeba6497aafc1b3263cf45d6c
  size: 81193310
  hash: md5
  path: data.csv
.dvc/cache
└── files
    └── md5
        └── a5
            └── 46111eeba6497aafc1b3263cf45d6c

4 directories, 1 file


`data.csv.dvc` only holds the file's md5, its size and its path: this small file is what git versions. The cache stores the data under their md5.

## 4. Save this version

We commit the files `dvc add` named, with the `v1` tag. `git push` only sends this metadata, and `dvc push` the data (81 MB), from the cache, to the storage.

In [9]:
!git add data.csv.dvc .gitignore
!git commit -m "Raw data"
!git tag v1

[main 8e57f0a] Raw data
 2 files changed, 6 insertions(+)
 create mode 100644 .gitignore
 create mode 100644 data.csv.dvc


In [10]:
!git push origin main v1
!dvc push

Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 8 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (4/4), 431 bytes | 431.00 KiB/s, done.
Total 4 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
To /tmp/remote.git
   753613a..8e57f0a  main -> main
 * [new tag]         v1 -> v1
Pushing
!
  0% Checking cache in '/tmp/dvc-store/files/md5'|   |0/? [00:00<?,    ?files/s]
                                                                                
!
  0% Checking cache in '/root/dvc-project/.dvc/cache/files/md5'| |0/? [00:00<?, 
                                                                                
!
  0%|          |Pushing to local                      0/1 [00:00<?,     ?file/s]
Pushing
1 file pushed


## 5. Modify the data

We sort the lines of `data.csv` in reverse order, then save this new version like the first one, with the `v2` tag.

In [11]:
!sort -r data.csv -o data.csv

In [12]:
!dvc add data.csv

⠋ Checking graph
Adding...                                                                       
!
                                                                                
!
  0% Checking cache in '/root/dvc-project/.dvc/cache/files/md5'| |0/? [00:00<?, 
                                                                                
!
  0%|          |Adding data.csv to cache              0/1 [00:00<?,     ?file/s]
                                                                                
!
  0%|          |Checking out /root/dvc-project/data.cs0/1 [00:00<?,    ?files/s]
100%|██████████|Checking out /root/dvc-project/da1/1 [00:00<00:00,  9.19files/s]
100% Adding...|████████████████████████████████████████|1/1 [00:00,  1.54file/s]

To track the changes with git, run:

	git add data.csv.dvc

To enable auto staging, run:

	dvc config core.autostage true


In [13]:
!cat data.csv.dvc
!tree .dvc/cache

outs:
- md5: 06d279899c8c10d4667a551b39950925
  size: 81193310
  hash: md5
  path: data.csv
.dvc/cache
└── files
    └── md5
        ├── 06
        │   └── d279899c8c10d4667a551b39950925
        └── a5
            └── 46111eeba6497aafc1b3263cf45d6c

5 directories, 2 files


In [14]:
!git add data.csv.dvc
!git commit -m "Data v2"
!git tag v2

[main 6006bf7] Data v2
 1 file changed, 1 insertion(+), 1 deletion(-)


In [15]:
!git push origin main v2
!dvc push

Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 8 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (3/3), 329 bytes | 329.00 KiB/s, done.
Total 3 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
To /tmp/remote.git
   8e57f0a..6006bf7  main -> main
 * [new tag]         v2 -> v2
Pushing
!
  0% Checking cache in '/tmp/dvc-store/files/md5'|   |0/? [00:00<?,    ?files/s]
                                                                                
!
  0% Checking cache in '/root/dvc-project/.dvc/cache/files/md5'| |0/? [00:00<?, 
                                                                                
!
  0%|          |Pushing to local                      0/1 [00:00<?,     ?file/s]
Pushing
1 file pushed


The md5 changed, not the size. The cache now holds both versions, each under its md5.

## 6. Back to the original data

The new version has a problem: we go back to the first one. `git restore --source v1` puts back the `data.csv.dvc` of `v1`, and `dvc checkout` the data it points to, from the cache.

In [16]:
!git restore --source v1 data.csv.dvc
!dvc checkout

Building workspace index                              |1.00 [00:00, 17.7entry/s]
Comparing indexes                                    |2.00 [00:00, 1.00kentry/s]
Applying changes                                      |1.00 [00:00,  10.2file/s]
M       data.csv


In [17]:
!cat data.csv.dvc

outs:
- md5: a546111eeba6497aafc1b3263cf45d6c
  size: 81193310
  hash: md5
  path: data.csv


In [18]:
!git add data.csv.dvc
!git commit -m "Back to data v1"
!git tag v3

[main e2105da] Back to data v1
 1 file changed, 1 insertion(+), 1 deletion(-)


In [19]:
!git push origin main v3
!dvc push

Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 8 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (3/3), 337 bytes | 337.00 KiB/s, done.
Total 3 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
To /tmp/remote.git
   6006bf7..e2105da  main -> main
 * [new tag]         v3 -> v3
Pushing
!
  0% Checking cache in '/tmp/dvc-store/files/md5'|   |0/? [00:00<?,    ?files/s]
Pushing
Everything is up to date.


`dvc push` has nothing to send: the `v1` data are already on the storage, under the same md5.

## 7. A pipeline stage

The `upper.py` script copies a file in uppercase. `dvc stage add` makes it a pipeline stage, with its dependencies (the script, the data) and its output, and `dvc repro` runs it and writes `dvc.lock`.

In [20]:
%%writefile upper.py
from pathlib import Path
from sys import argv

text = Path(argv[1]).read_text(encoding="utf-8")
Path(argv[2]).write_text(text.upper(), encoding="utf-8")

Writing upper.py


In [21]:
!dvc stage add --name uppercase --deps upper.py --deps data.csv --outs data-upper.csv python upper.py data.csv data-upper.csv

Added stage 'uppercase' in 'dvc.yaml'

To track the changes with git, run:

	git add dvc.yaml .gitignore

To enable auto staging, run:

	dvc config core.autostage true


In [22]:
!dvc repro

'data.csv.dvc' didn't change, skipping
Running stage 'uppercase':
> python upper.py data.csv data-upper.csv
Generating lock file 'dvc.lock'
Updating lock file 'dvc.lock'

To track the changes with git, run:

	git add dvc.lock

To enable auto staging, run:

	dvc config core.autostage true
Use `dvc push` to send your updates to remote storage.


In [23]:
!git add upper.py dvc.yaml dvc.lock .gitignore
!git commit -m "Uppercase pipeline"
!git push origin main
!dvc push

[main cf54a70] Uppercase pipeline
 4 files changed, 32 insertions(+)
 create mode 100644 dvc.lock
 create mode 100644 dvc.yaml
 create mode 100644 upper.py
Enumerating objects: 8, done.
Counting objects: 100% (8/8), done.
Delta compression using up to 8 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (6/6), 864 bytes | 864.00 KiB/s, done.
Total 6 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
To /tmp/remote.git
   e2105da..cf54a70  main -> main
Pushing
!
  0% Checking cache in '/tmp/dvc-store/files/md5'|   |0/? [00:00<?,    ?files/s]
                                                                                
!
  0% Checking cache in '/root/dvc-project/.dvc/cache/files/md5'| |0/? [00:00<?, 
                                                                                
!
  0%|          |Pushing to local                      0/1 [00:00<?,     ?file/s]
Pushing
1 file pushed


In [24]:
!dvc repro
!cat dvc.lock

'data.csv.dvc' didn't change, skipping
Stage 'uppercase' didn't change, skipping
Data and pipelines are up to date.
schema: '2.0'
stages:
  uppercase:
    cmd: python upper.py data.csv data-upper.csv
    deps:
    - path: data.csv
      hash: md5
      md5: a546111eeba6497aafc1b3263cf45d6c
      size: 81193310
    - path: upper.py
      hash: md5
      md5: 8ba832b8c3bb0ed0f639f87bfb806dc0
      size: 153
    outs:
    - path: data-upper.csv
      hash: md5
      md5: 2a6ce2d726d43441b0b12c6e3d35c7a1
      size: 81059022


The second `dvc repro` skips the stage: nothing changed since the md5 in `dvc.lock`. `dvc.lock` says exactly from what and how `data-upper.csv` was produced: this is data lineage.

## 8. On another machine

A colleague joins the project: she clones the repository and retrieves the data with `dvc pull`. With DagsHub, she would first give the clone her credentials, which git didn't pass on.

In [25]:
!git clone /tmp/remote.git ~/other-machine

Cloning into '/root/other-machine'...
done.


In [26]:
%cd ~/other-machine

/root/other-machine


In [27]:
!dvc pull

Fetching
!
  0% Checking cache in '/root/other-machine/.dvc/cache/files/md5'| |0/? [00:00<?
                                                                                
!
  0% Checking cache in '/tmp/dvc-store/files/md5'|   |0/? [00:00<?,    ?files/s]
                                                                                
!
  0%|          |Fetching from local                   0/2 [00:00<?,     ?file/s]
100%|██████████|Fetching from local               2/2 [00:00<00:00, 12.24file/s]
Fetching
Building workspace index                              |0.00 [00:00,    ?entry/s]
Comparing indexes                                    |3.00 [00:00, 1.01kentry/s]
Applying changes                                      |2.00 [00:00,  11.8file/s]
A       data-upper.csv
A       data.csv
2 files fetched and 2 files added


In [28]:
!ls
!dvc status
!dvc repro

data-upper.csv	data.csv  data.csv.dvc	dvc.lock  dvc.yaml  upper.py
Data and pipelines are up to date.
'data.csv.dvc' didn't change, skipping
Stage 'uppercase' didn't change, skipping
Data and pipelines are up to date.


The clone has `data.csv` (in its `v3` version) and `data-upper.csv`, without running the script again: the data and the pipeline are up to date.